# Generate Coordinate CSV

This notebook generates a `geo_coordinates.csv` file directly from the GeoTIFF images in the dataset.

The notebook scans the following folders:

- `train_A`
- `test_A`

For each GeoTIFF image, it:

1. Reads the geometric center of the raster using **GDAL**.
2. Converts the center coordinate from the source CRS to **EPSG:4326**.
3. Converts longitude and latitude to unit-sphere Cartesian coordinates (`xs`, `ys`, `zs`).
4. Writes the following fields to `geo_coordinates.csv`:

`filename, xs, ys, zs, longitude, latitude, source_x, source_y, source_crs, folder`

By default, the dataset is read from the project’s datasets/ directory. Adjust DATAROOT if needed. Use FOLDERS = ["train_A", "test_A"] for training and testing, or FOLDERS = ["test_A"] for a test-only dataset.

The output CSV file is saved automatically as:

`<DATAROOT>/geo_coordinates.csv`

Before writing the CSV, the notebook performs basic validation checks, including:

- total row count,
- records per folder,
- duplicate filenames across all selected folders,
- and verification that `xs² + ys² + zs² ≈ 1`.

The generated CSV is intended to provide geographic coordinate metadata for the coordinate-conditioned pix2pixHD workflow.


In [ ]:
from pathlib import Path
import csv
import numpy as np
from osgeo import gdal, osr


# ============================================================
# USER SETTING
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train.py").is_file(), "Set PROJECT_ROOT to the project directory."
DATAROOT = PROJECT_ROOT / "datasets"
OUTPUT_CSV = DATAROOT / "geo_coordinates.csv"
FOLDERS = ["train_A", "test_A"]  # For test-only datasets, use ["test_A"].


# ============================================================
# CSV STRUCTURE
# ============================================================

FIELDNAMES = [
    "filename",
    "xs",
    "ys",
    "zs",
    "longitude",
    "latitude",
    "source_x",
    "source_y",
    "source_crs",
    "folder",
]


# ============================================================
# HELPERS
# ============================================================

def get_epsg_string(srs):
    """Return a CRS string such as EPSG:32632."""
    if srs is None:
        raise ValueError("Raster CRS is missing.")

    srs = srs.Clone()

    try:
        srs.AutoIdentifyEPSG()
    except Exception:
        pass

    for node in (None, "PROJCS", "GEOGCS"):
        authority_name = srs.GetAuthorityName(node)
        authority_code = srs.GetAuthorityCode(node)

        if authority_name and authority_code:
            return f"{authority_name}:{authority_code}"

    raise ValueError(
        "Could not identify an EPSG code from raster CRS:\n"
        + srs.ExportToPrettyWkt()
    )


def calculate_coordinates(tif_path):
    """
    Calculate the geographic centre of one GeoTIFF with GDAL and
    convert longitude/latitude to unit-sphere Cartesian coordinates.
    """
    dataset = gdal.Open(str(tif_path), gdal.GA_ReadOnly)

    if dataset is None:
        raise RuntimeError(f"GDAL could not open: {tif_path}")

    try:
        width = dataset.RasterXSize
        height = dataset.RasterYSize

        geotransform = dataset.GetGeoTransform(can_return_null=True)

        if geotransform is None:
            raise ValueError(f"GeoTransform is missing: {tif_path}")

        # Geometric raster centre.
        col = width / 2.0
        row = height / 2.0

        source_x = (
            geotransform[0]
            + col * geotransform[1]
            + row * geotransform[2]
        )

        source_y = (
            geotransform[3]
            + col * geotransform[4]
            + row * geotransform[5]
        )

        source_srs = dataset.GetSpatialRef()

        if source_srs is None:
            projection_wkt = dataset.GetProjection()

            if not projection_wkt:
                raise ValueError(f"CRS is missing: {tif_path}")

            source_srs = osr.SpatialReference()
            source_srs.ImportFromWkt(projection_wkt)

        source_crs = get_epsg_string(source_srs)

        # GDAL 3+ can use authority-compliant axis order.
        # Force the conventional GIS order:
        # x = longitude/easting, y = latitude/northing.
        source_srs = source_srs.Clone()
        source_srs.SetAxisMappingStrategy(
            osr.OAMS_TRADITIONAL_GIS_ORDER
        )

        target_srs = osr.SpatialReference()
        target_srs.ImportFromEPSG(4326)
        target_srs.SetAxisMappingStrategy(
            osr.OAMS_TRADITIONAL_GIS_ORDER
        )

        transformer = osr.CoordinateTransformation(
            source_srs,
            target_srs,
        )

        longitude, latitude, _ = transformer.TransformPoint(
            source_x,
            source_y,
        )

    finally:
        dataset = None

    # Longitude/latitude -> Cartesian coordinates on the unit sphere.
    lon_rad = np.deg2rad(longitude)
    lat_rad = np.deg2rad(latitude)

    xs = np.cos(lat_rad) * np.cos(lon_rad)
    ys = np.cos(lat_rad) * np.sin(lon_rad)
    zs = np.sin(lat_rad)

    return {
        "xs": float(xs),
        "ys": float(ys),
        "zs": float(zs),
        "longitude": float(longitude),
        "latitude": float(latitude),
        "source_x": float(source_x),
        "source_y": float(source_y),
        "source_crs": source_crs,
    }


# ============================================================
# GENERATE CSV
# ============================================================

if not DATAROOT.is_dir():
    raise FileNotFoundError(f"Dataset root not found: {DATAROOT}")

rows = []

for folder_name in FOLDERS:
    folder_path = DATAROOT / folder_name

    if not folder_path.is_dir():
        raise FileNotFoundError(
            f"Dataset folder not found: {folder_path}"
        )

    tif_files = sorted(
        [
            path
            for path in folder_path.iterdir()
            if path.is_file()
            and path.suffix.lower() in (".tif", ".tiff")
        ],
        key=lambda path: path.name,
    )

    print(f"{folder_name}: {len(tif_files)} GeoTIFF")

    for index, tif_path in enumerate(tif_files, start=1):
        coord = calculate_coordinates(tif_path)

        rows.append(
            {
                "filename": tif_path.name,
                "xs": coord["xs"],
                "ys": coord["ys"],
                "zs": coord["zs"],
                "longitude": coord["longitude"],
                "latitude": coord["latitude"],
                "source_x": coord["source_x"],
                "source_y": coord["source_y"],
                "source_crs": coord["source_crs"],
                "folder": folder_name,
            }
        )

        if index % 1000 == 0 or index == len(tif_files):
            print(
                f"  {folder_name}: "
                f"{index}/{len(tif_files)} processed"
            )


# BASIC VALIDATION
print()
print("Total records:", len(rows))

folder_counts = {}
duplicate_keys = set()
seen_keys = set()
max_norm_error = 0.0

for row in rows:
    folder_counts[row["folder"]] = folder_counts.get(row["folder"], 0) + 1
    key = row["filename"]

    if key in seen_keys:
        duplicate_keys.add(key)
    seen_keys.add(key)

    norm = row["xs"] ** 2 + row["ys"] ** 2 + row["zs"] ** 2
    max_norm_error = max(max_norm_error, abs(norm - 1.0))

print("Records by folder:")
for folder_name in FOLDERS:
    print(f"  {folder_name}: {folder_counts.get(folder_name, 0)}")

print("Duplicate filenames:", len(duplicate_keys))
print("Maximum |xs² + ys² + zs² - 1|:", f"{max_norm_error:.16e}")

if duplicate_keys:
    raise RuntimeError("Duplicate filenames were found across the selected folders.")

if max_norm_error > 1e-12:
    raise RuntimeError("Unit-sphere XYZ validation failed.")

print()
print("Validation passed.")

# Write the CSV after validation succeeds.
with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(rows)

print("CSV created:", OUTPUT_CSV)